In [ ]:
# Machine Learning on Human Carbonic Anhydrase II Inhibition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME/blob/main/YOUR_FILENAME.ipynb)

**Author:** [Babak Mamnoon/GitHub]
**Project:** Classification of hCA II Inhibitors using ChEMBL data and ECFP4 Fingerprints.

In [ ]:
# Installing Cheminformatics and ML libraries
!pip install rdkit chembl_webresource_client xgboost -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, 
                             confusion_matrix, ConfusionMatrixDisplay, 
                             average_precision_score, accuracy_score)

print("Libraries imported successfully.")

Step A — Data Acquisition & Exploration

In [ ]:
print("--- Step A: Extracting ChEMBL Dataset ---")

# Target ID for Human Carbonic Anhydrase II
TARGET_ID = 'CHEMBL205' 

activity = new_client.activity
res = activity.filter(target_chembl_id=TARGET_ID).filter(standard_type="IC50")

# Convert to DataFrame
df_raw = pd.DataFrame.from_dict(res)

# Data Cleaning
df = df_raw.dropna(subset=['standard_value', 'canonical_smiles'])
df['standard_value'] = df['standard_value'].astype(float)

# Define Binary Classes: Active (1) if IC50 < 1000 nM, else Inactive (0)
df['class'] = df['standard_value'].apply(lambda x: 1 if x < 1000 else 0)

print(f"Total entries retrieved: {len(df_raw)}")
print(f"Cleaned entries (valid SMILES/IC50): {len(df)}")
print("-" * 30)
print("Data Preview (First 5 Compounds):")
display(df[['molecule_chembl_id', 'canonical_smiles', 'standard_value', 'class']].head())

Step B — Molecular Featurization (ECFP4)

In [ ]:
print("--- Step B: Converting SMILES to Morgan Fingerprints ---")

def generate_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        # Radius 2 = ECFP4; 2048-bit vector
        return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048))
    return None

df['fp'] = df['canonical_smiles'].apply(generate_fp)
df = df.dropna(subset=['fp'])

# Prepare X (features) and y (target)
X = np.stack(df['fp'].values)
y = df['class'].values

# Stratified split to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Featurization complete.")
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Step C — Model Training & Individual Evaluation

In [ ]:
print("--- Step C: Training & Plotting Individual Model Metrics ---")

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    "SVM (RBF)": SVC(kernel='rbf', probability=True, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

performance_log = {}

for name, model in models.items():
    print(f"Processing {name}...")
    model.fit(X_train, y_train)
    y_probs = model.predict_proba(X_test)[:, 1]
    y_preds = model.predict(X_test)
    
    # Metrics
    fpr, tpr, _ = roc_curve(y_test, y_probs)
    roc_auc = auc(fpr, tpr)
    precision, recall, _ = precision_recall_curve(y_test, y_probs)
    avg_prec = average_precision_score(y_test, y_probs)
    
    # Individual Plots
    fig, ax = plt.subplots(1, 3, figsize=(18, 4))
    
    # ROC Curve
    ax[0].plot(fpr, tpr, color='blue', label=f'AUC = {roc_auc:.3f}')
    ax[0].plot([0, 1], [0, 1], 'k--')
    ax[0].set_title(f'{name} ROC')
    ax[0].legend()
    
    # Precision-Recall Curve
    ax[1].plot(recall, precision, color='purple', label=f'AP = {avg_prec:.3f}')
    ax[1].set_title(f'{name} PR Curve')
    ax[1].legend()
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax[2])
    ax[2].set_title(f'{name} Confusion Matrix')
    
    plt.show()
    
    performance_log[name] = {"ROC-AUC": roc_auc, "Accuracy": accuracy_score(y_test, y_preds)}

Final Comparison & Conclusion

In [ ]:
# Sorting models by ROC-AUC
sorted_summary = sorted(performance_log.items(), key=lambda x: x[1]['ROC-AUC'], reverse=True)

print("\n" + "="*50)
print("FINAL SCIENTIFIC CONCLUSION")
print("="*50)

for i, (name, stats) in enumerate(sorted_summary):
    print(f"{i+1}. {name}: ROC-AUC = {stats['ROC-AUC']:.3f}, Accuracy = {stats['Accuracy']:.1%}")

best_model_name = sorted_summary[0][0]
best_auc = sorted_summary[0][1]['ROC-AUC']

print(f"\nSummary: The {best_model_name} model emerged as the top performer with an AUC of {best_auc:.3f}.")
print("This suggests that non-linear ensemble methods are highly effective at identifying")
print("potency-determining structural features in hCA II inhibitors.")
print("="*50)